## YOLO

- Python을 기반하는 영상인식 라이브러리
- 라즈베리파이 사용 가능
- 라즈베리파이 5 기준
    - YOLOv5 : 10~20FPS
    - YOLOv8n : 8~17FPS
    - YOLOv8s : 3~7FPS

- YOLO 사용시 대부분 Jetson Nano라는 NVIDIA 제품 사용

### 객체 탐지
- Object Detection
    - 이미지, 영상(한 프레임 이미지) 위치와 클래스 모두 예측
    - x,y,w,h 영역안에 물체를 감지
    - 자율주행, 로봇, CCTV분석등 활용

- 원리
    - 물체후보영역 추출
    - 각 영역 분류
    - 위 원리를 더 세분화, 그리드화 시킨 알고리즘 활용 : YOLO

### YOLO 설치

- Numpy, opencv-python, Pytorch, Pytorch Vision 라이브러리 필수
- 미리 설치 권고
- Pytorch는 GPU버전 미리 설치

### 1. 라이브러리 로드

In [ ]:
from ultralytics import YOLO

- 최초 로드시 C:\Users\User\AppData\Roaming\Ultralytics\settings.json 생성
- settings.json 파일에 datasets_dir, weights_dir, runs_dir 수정

### 2. 사전학습 모델 다운로드

In [ ]:
# n(Nano), s(Small), m(Medium), l(Large), x(eXtra Large)
model = YOLO('yolo11n.pt')    # yolo8n.pt. yolo11n.pt

### 3. 이미지 예측

In [ ]:
results = model('./bus.jpg')

### 4. 결과확인

In [ ]:
results

- yolo11n 은 총 80개 물체 인식
{0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant', 59: 'bed', 60: 'dining table', 61: 'toilet', 62: 'tv', 63: 'laptop', 64: 'mouse', 65: 'remote', 66: 'keyboard', 67: 'cell phone', 68: 'microwave', 69: 'oven', 70: 'toaster', 71: 'sink', 72: 'refrigerator', 73: 'book', 74: 'clock', 75: 'vase', 76: 'scissors', 77: 'teddy bear', 78: 'hair drier', 79: 'toothbrush'}

In [6]:
results[0].show()

### 5. 결과 이미지 저장

In [ ]:
results[0].save(filename='result01.jpg')

### 6. 데이터폴더 전체 인식

In [ ]:
results = model('./data/')

for i, result in enumerate(results):
    result.save(filename = f'result_{i:02d}.jpg')

### 7. 동영상 인식

In [ ]:
model.predict(
    source ='./sample02.mp4',
    show=True,
    save =True,
    conf = 0.5
)

### 8. 동영상 플레이하면서 물체인식

In [ ]:
# 영상출력 기본
import cv2
import time
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

# cap = cv2.VideoCapture('./sample02.mp4')
cap = cv2.VideoCapture(0)

# FPS 계산용
prev_time = 0

while True:
    # 프레임 읽기
    ret, frame = cap.read()
    if not ret: break   # 더이상 영상이 없으면 종료

    # YOLO 영상 추출
    results = model(frame, conf = 0.5)
    pred_frame = results[0].plot()

    curr_time = time.time()
    fps = 1 / (curr_time - prev_time)
    prev_time = curr_time

    # FPS 출력
    cv2.putText(pred_frame,
                f'FPS : {fps:.1f}',
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 255, 0),
                2)

    cv2.imshow('YOLO Player', pred_frame,)

    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

### 9. 침입탐지

In [ ]:
import cv2
import time
from ultralytics import YOLO
import winsound

model = YOLO('yolo11n.pt')
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret: break   # 더이상 영상이 없으면 종료
    # YOLO 영상 추출
    results = model(frame, conf = 0.5)
    detected = False

    # 사람 인식
    for box in results[0].boxes:
        cls_id = int(box.cls[0])
        label = model.names[cls_id]

        if label == 'person':
            detected = True

    pred_frame = results[0].plot()

    if detected:
        cv2.putText(
            pred_frame,
            'Person Detected',
            (20, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 0, 255),
            2
        )
        winsound.Beep(1000, 300)    # 경고음

    cv2.imshow('Detection Alarm', pred_frame,)

    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

### 10. COCO 객체

- Common Object in COntext : 컴퓨터 비전에서 가장 유명한 데이터셋 중 하나
- YOLO 기본모델은 COCO DataSet만 인식 : 80가지

- 80가지 이외의 물체 인식하려면 
    - 직접 학습시킨 모델 생성
    - 타 사이트에서 제공하는 모델 사용 : https://huggingface.co/

### 11. 화재 감지

In [ ]:
import cv2
import time
from ultralytics import YOLO

model = YOLO('./firedetect-11s.pt')
cap = cv2.VideoCapture('./sample03.mp4')

while True:
    ret, frame = cap.read()
    if not ret: break

    results = model(frame)
    pred_frame = results[0].plot()

    cv2.imshow('Fire Detection', pred_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()